# Current prepared XYZ: one-day example

Generate a fresh waveform while reusing matched retarded geometry and TDI
operators from this checkout. Use Python 3.12+ with this repository installed
as `pip install -e ".[orbits,notebook]"`. Restart the kernel and run all cells.

In [ ]:
from pathlib import Path
from dataclasses import replace
from time import perf_counter
import hashlib
import importlib.metadata as metadata
import inspect
import json
import os
import platform
import subprocess
import sys

# Load the current checkout from any directory inside this repository.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/egb_jax_eccentric").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from a directory inside the repository")
sys.path.insert(0, str(ROOT / "src"))
os.environ["JAX_PLATFORMS"] = "cpu"
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)

from egb_jax_eccentric import (
    EccentricBinaryParams, LINK_LABELS, default_lisaorbits,
    state_from_lisaorbits, precompute_jax_link_geometry,
    eccentric_links_jax, prepare_xyz_from_links, xyz_from_links,
)
assert Path(inspect.getfile(prepare_xyz_from_links)).is_relative_to(ROOT), (
    "Restart the kernel to load this checkout")

environment = {
    "python": platform.python_version(), "platform": platform.platform(),
    "devices": [str(device) for device in jax.devices()],
    "jax_float64": jax.config.x64_enabled,
    "versions": {name: metadata.version(name) for name in
                 ("numpy", "scipy", "jax", "jaxlib", "pytdi", "lisaorbits")},
    "tdi_source_sha256": hashlib.sha256(
        Path(inspect.getfile(prepare_xyz_from_links)).read_bytes()).hexdigest(),
}
print(json.dumps(environment, indent=2))

def build_cache(times, orbit, measurement_order=3, delay_order=3):
    tick = perf_counter()
    state = state_from_lisaorbits(orbit, times, frame="ecliptic")
    geometry = precompute_jax_link_geometry(state, orbits=orbit, frame="ecliptic")
    delays = {label: geometry["length"][i]
              for i, label in enumerate(LINK_LABELS)}
    geometry_s = perf_counter() - tick
    tick = perf_counter()
    tdi = prepare_xyz_from_links(state, generation=2,
        measurement_order=measurement_order, delay_order=delay_order,
        link_delays=delays)
    tdi_s = perf_counter() - tick
    return state, geometry, delays, tdi, np.array([geometry_s, tdi_s])

def prepared_waveform(source, geometry, tdi):
    # This public wrapper materializes host arrays, synchronizing JAX.
    links = eccentric_links_jax(source, geometry, batch_size=1,
        physics_mode="1pn_periastron", evolution_mode="fixed")
    return tdi(links)

def make_source(frequency_hz=1e-3, eccentricity=.3):
    return EccentricBinaryParams(
        mean_motion=np.pi * frequency_hz, eccentricity=eccentricity,
        m1_solar=.6, m2_solar=.4, distance_m=3.085677581491367e19,
        beta=.2, lambda_=.4, psi=.3, inclination=.8, phi0=.2, fdot=0.,
    )

In [ ]:
dt = 5.0
times = np.arange(0., 86400., dt)
orbit = default_lisaorbits("esa-trailing")
state, geometry, delays, tdi, setup = build_cache(times, orbit)
source = make_source()
print(f"Geometry preparation: {setup[0]:.4f} s")
print(f"TDI operator preparation: {setup[1]:.4f} s")

tick = perf_counter()
xyz = prepared_waveform(source, geometry, tdi)
first_call_s = perf_counter() - tick
print(f"First waveform call (includes JAX compilation): {first_call_s:.4f} s")

In [ ]:
# Changed source, unchanged detector cache. Initial JIT work is complete.
timings = []
for _ in range(5):
    tick = perf_counter()
    xyz = prepared_waveform(replace(source, eccentricity=.4), geometry, tdi)
    timings.append(perf_counter() - tick)
    assert all(np.isfinite(values).all() for values in xyz.values())

print({channel: values.shape for channel, values in xyz.items()})
print(f"Warm waveform + TDI median: {np.median(timings):.4f} s")
print(f"Warm repeat range: {min(timings):.4f}–{max(timings):.4f} s")
print("Warm times exclude cache preparation, first-call compilation and checks.")

These are CPU float64 **fixed-evolution** waveforms with 1PN periastron
advance, native ESA-trailing retarded link geometry in ecliptic J2000, and
frequency-unit generation-2 TDI. The same light travel times feed links and
TDI. Geometry/operator caches are independent of source parameters.

The 5 s cadence and interpolation orders 3/3 specify this timing workload;
they are not an accuracy-convergence claim or the full referee fitting-factor
search. No waveform files are saved. Rebuild the cache when the grid, orbit
or interpolation settings change.

`xyz` contains dimensionless fractional-frequency X/Y/Z arrays. This short
calling example includes finite-grid edge samples; the full-year notebook
uses padding and discards those edges.